# Grid Runner — Google Colab

Runs a filtered subset of the 324-experiment grid for the **adaptive-imbalance-tsc** AAAI 2026 paper.

**Before you start:**
1. Set the Runtime type to **GPU** (Runtime → Change runtime type → T4 GPU).
2. Mount Google Drive (cell 2) so results survive runtime restarts.
3. Paste your W&B API key in cell 3 (optional).

In [ ]:
# ── Cell 1: Setup ──────────────────────────────────────────────────────────
# Install dependencies
!pip install torch aeon wfdb wandb tqdm pyyaml scikit-learn --quiet

import os, sys

REPO_DIR = "/content/adaptive-imbalance-tsc"

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/YOUR_USERNAME/adaptive-imbalance-tsc.git {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)
print("Working directory:", os.getcwd())

In [ ]:
# ── Cell 2: Mount Google Drive ─────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_RESULTS = "/content/drive/MyDrive/adaptive-imbalance-tsc/results"
os.makedirs(DRIVE_RESULTS, exist_ok=True)

# Symlink so run_experiment.py writes directly to Drive
LOCAL_RESULTS = os.path.join(REPO_DIR, "experiments", "results")
os.makedirs(os.path.dirname(LOCAL_RESULTS), exist_ok=True)

if not os.path.islink(LOCAL_RESULTS):
    if os.path.exists(LOCAL_RESULTS):
        import shutil
        shutil.rmtree(LOCAL_RESULTS)
    os.symlink(DRIVE_RESULTS, LOCAL_RESULTS)

print(f"Results will be saved to: {DRIVE_RESULTS}")

In [ ]:
# ── Cell 3: W&B API key (optional) ────────────────────────────────────────
# Paste your W&B API key below to enable experiment tracking.
# Leave empty string to skip W&B logging.
import os

WANDB_API_KEY = ""  # <-- paste your key here, e.g. "abc123..."

if WANDB_API_KEY:
    os.environ["WANDB_API_KEY"] = WANDB_API_KEY
    !wandb login {WANDB_API_KEY} --relogin --quiet
    print("W&B logged in.")
else:
    print("W&B skipped (no key provided). Set wandb.enabled=false in configs or provide key above.")

In [ ]:
# ── Cell 4: Filter selector ────────────────────────────────────────────────
# Adjust these lists to control which experiments run on this session.

METHOD_FILTER  = ['class_balanced', 'ldam', 'logit_adj']   # change as needed
DATASET_FILTER = []   # empty = all datasets
ARCH_FILTER    = []   # empty = all architectures

print("Methods  :", METHOD_FILTER  or "ALL")
print("Datasets :", DATASET_FILTER or "ALL")
print("Archs    :", ARCH_FILTER    or "ALL")

In [ ]:
# ── Cell 5: Generate configs (idempotent) + run grid ──────────────────────
import os
os.chdir(REPO_DIR)

# Generate all 324 configs (safe to run multiple times)
!python scripts/generate_configs.py --output_dir configs/experiments

# Build CLI args
filter_args = ""
if METHOD_FILTER:
    filter_args += " --filter_method " + " ".join(METHOD_FILTER)
if DATASET_FILTER:
    filter_args += " --filter_dataset " + " ".join(DATASET_FILTER)
if ARCH_FILTER:
    filter_args += " --filter_arch " + " ".join(ARCH_FILTER)

cmd = f"python scripts/run_grid.py --config_dir configs/experiments --skip_completed {filter_args}"
print("Running:", cmd)
!{cmd}

In [ ]:
# ── Cell 6: Sync results back to Drive ────────────────────────────────────
# Results are already being written to Drive via the symlink.
# This cell copies any extra artefacts (checkpoints, logs).

import shutil, os
DRIVE_CHECKPOINTS = "/content/drive/MyDrive/adaptive-imbalance-tsc/checkpoints"
os.makedirs(DRIVE_CHECKPOINTS, exist_ok=True)

LOCAL_CKPT = os.path.join(REPO_DIR, "experiments", "checkpoints")
if os.path.isdir(LOCAL_CKPT):
    shutil.copytree(LOCAL_CKPT, DRIVE_CHECKPOINTS, dirs_exist_ok=True)
    print("Checkpoints synced to Drive.")

# Copy run log
run_log = os.path.join(REPO_DIR, "experiments", "run_log.csv")
if os.path.exists(run_log):
    shutil.copy(run_log, "/content/drive/MyDrive/adaptive-imbalance-tsc/run_log.csv")
    print("run_log.csv synced.")

print("Sync complete.")

In [ ]:
# ── Cell 7: Quick results table ────────────────────────────────────────────
import os
os.chdir(REPO_DIR)

!python scripts/consolidate_results.py \
    --results_dir experiments/results \
    --output experiments/raw_results.csv

import pandas as pd
try:
    df = pd.read_csv("experiments/raw_results.csv")
    print(f"\nTotal results: {len(df)}")
    display(df[['experiment_name','method','dataset','architecture','seed',
                'macro_f1','balanced_accuracy','g_mean']].sort_values('macro_f1', ascending=False).head(20))
except FileNotFoundError:
    print("No results yet.")